# Лабораторная работа 6. Решающие деревья и композиции алгоритмов

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 5 |
| Опора на лекции | лекция 5: решающее дерево (опр. 5.1), критерии информативности (опр. 5.2–5.4), прирост информации (опр. 5.5) и его неотрицательность (утв. 5.6), стрижка по цене сложности (опр. 5.8), бэггинг (опр. 5.11), разброс усреднённого предсказания (утв. 5.12), AdaBoost (теорема 5.15) |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Разобраться, почему деревья строят по энтропии и Джини, а не по доле ошибок; увидеть, что бэггинг и бустинг борются с разными слагаемыми разложения из занятия 5; научиться корректно измерять важность признаков и понять, почему встроенная важность систематически врёт.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab06_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import make_moons
from sklearn.ensemble import (AdaBoostClassifier, BaggingRegressor,
                              GradientBoostingClassifier, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=6)
describe_variant(variant)

---
# Часть 1. Почему не доля ошибок

Определения 5.2–5.4: для множества $R$ с долями классов $p_k$

$$
H(R) = -\sum_k p_k\ln p_k, \qquad \mathrm{Gini}(R) = 1 - \sum_k p_k^2 .
$$

Прирост информации при разбиении $R = R_L\sqcup R_R$:

$$
\mathrm{Gain} = \Phi(R) - \frac{|R_L|}{|R|}\Phi(R_L) - \frac{|R_R|}{|R|}\Phi(R_R).
$$

Утверждение 5.6 гарантирует $\mathrm{Gain}\ge0$ для энтропии и Джини — это
следствие их **строгой** вогнутости. Доля ошибок $\min(p, 1-p)$ тоже вогнута,
но не строго. Посмотрим, к чему это приводит.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def entropy(y):
    """H(R) = -sum p_k ln p_k (соглашение 0 ln 0 = 0: отбросьте нулевые p_k)."""
    raise NotImplementedError


def gini(y):
    """Gini(R) = 1 - sum p_k^2."""
    raise NotImplementedError


def gain(y, mask, phi):
    """Прирост информации при разбиении множества по булевой маске.

    Формула выше; если одна из частей пуста -- прирост нулевой.
    """
    raise NotImplementedError

In [ ]:
p = np.linspace(1e-9, 1 - 1e-9, 400)
fig, ax = plt.subplots()
ax.plot(p, -(p * np.log(p) + (1 - p) * np.log(1 - p)), lw=2, label="энтропия")
ax.plot(p, 2 * p * (1 - p), lw=2, label="Джини")
ax.plot(p, np.minimum(p, 1 - p), lw=2, ls="--", label=r"доля ошибок $\min(p,1-p)$")
ax.set_xlabel("$p_1$ — доля первого класса"); ax.set_ylabel(r"$\Phi$")
ax.set_title("Критерии информативности (два класса)"); ax.legend()
plt.tight_layout(); plt.show()
print("доля ошибок кусочно-линейна: вогнута, но НЕ строго")

In [ ]:
# Проверяем утверждение 5.6 на 20000 случайных разбиений
err_rate = lambda yy: 1 - np.max(np.bincount(yy, minlength=3)) / len(yy)
gen = np.random.default_rng(RANDOM_STATE)

for name, phi in [("энтропия", entropy), ("Джини", gini), ("доля ошибок", err_rate)]:
    vals = []
    for _ in range(20_000):
        n = int(gen.integers(4, 40))
        y_r = gen.integers(0, 3, n)
        mask = gen.random(n) < gen.uniform(0.15, 0.85)
        if mask.all() or (~mask).all():
            continue
        vals.append(gain(y_r, mask, phi))
    vals = np.array(vals)
    print(f"{name:14s}: min Gain = {vals.min():+.1e} | доля отрицательных "
          f"{np.mean(vals < -1e-12):.4f} | доля НУЛЕВЫХ {np.mean(vals < 1e-12):.4f}")

> **Вывод.** У какого критерия нулевой прирост встречается чаще всего и почему? Что это значит для жадного алгоритма построения дерева?
>
> *(ваш ответ здесь)*

---
# Часть 2. Дерево: ступенчатая граница и стрижка

Дерево строится жадно: в вершине перебираются все признаки и пороги, берётся
расщепление с максимальным приростом. Каждый предикат имеет вид
$[x_j \le t]$ — отсюда важное следствие для формы границы.

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=RANDOM_STATE)
Xa, Xv, ya, yv = train_test_split(Xm, ym, test_size=0.35, random_state=RANDOM_STATE)


def plot_boundary(ax, model, X, y, title):
    g1, g2 = np.meshgrid(np.linspace(X[:, 0].min() - .4, X[:, 0].max() + .4, 250),
                         np.linspace(X[:, 1].min() - .4, X[:, 1].max() + .4, 250))
    Z = model.predict(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    ax.contourf(g1, g2, Z, levels=[-.5, .5, 1.5], colors=["#cfe8e4", "#f6ddc4"])
    ax.scatter(*X[y == 0].T, s=10, marker="o"); ax.scatter(*X[y == 1].T, s=10, marker="s")
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, depth in zip(axes, [1, 3, 6, None]):
    m = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(Xa, ya)
    plot_boundary(ax, m, Xa, ya, f"глубина {depth}, листьев {m.get_n_leaves()}\n"
                                 f"обучение {m.score(Xa, ya):.2f}, контроль {m.score(Xv, yv):.2f}")
plt.tight_layout(); plt.show()

### Стрижка по цене сложности

Определение 5.8: $Q_\alpha(T) = Q(T, X^\ell) + \alpha\,|\mathrm{leaves}(T)|$.
Это в точности принцип SRM из лекции 4, применённый к вложенному семейству
поддеревьев: $\alpha$ штрафует сложность и подбирается скользящим контролем.

> **Напоминание — SRM.** Structural Risk Minimization, минимизация структурного риска (опр. 4.14):
> семейство моделей разбивают на вложенную цепочку
> $A_1\subset A_2\subset\dots$ по возрастанию сложности и минимизируют не
> эмпирический риск, а его сумму со штрафом за сложность. Мы это уже делали
> трижды: степень полинома (занятие 2), $\lambda$ в Ridge и LASSO (занятие 3),
> теперь $\alpha$ и число листьев. Каждый раз одна и та же схема — «ошибка плюс
> плата за сложность», меняется только мера сложности.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

full = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(Xa, ya)
alphas = np.unique(full.cost_complexity_pruning_path(Xa, ya).ccp_alphas)
alphas = alphas[alphas >= 0][:: max(1, len(alphas) // 30)]
cv6 = StratifiedKFold(5, shuffle=True, random_state=0)

# TODO (2-3 строки): для каждого alpha оцените дерево скользящим контролем
#   (DecisionTreeClassifier(ccp_alpha=a), cross_val_score) и выберите alpha*
scores = ...
a_best = ...
print(f"alpha* = {a_best:.5f}")

In [ ]:
pruned = DecisionTreeClassifier(random_state=0, ccp_alpha=a_best).fit(Xa, ya)
leaves = [DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(Xa, ya).get_n_leaves()
          for a in alphas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(alphas, leaves, "o-", lw=2, color="#128C7E")
ax1.set_xscale("symlog", linthresh=1e-4); ax1.set_xlabel(r"$\alpha$")
ax1.set_ylabel("число листьев"); ax1.set_title("Стрижка по цене сложности")
ax2.plot(alphas, scores, "o-", lw=2)
ax2.axvline(a_best, ls="--", color="#C97A2B", label=fr"$\alpha^* = {a_best:.4f}$")
ax2.set_xscale("symlog", linthresh=1e-4); ax2.set_xlabel(r"$\alpha$")
ax2.set_ylabel("точность по контролю"); ax2.legend(); ax2.set_title(r"Выбор $\alpha$")
plt.tight_layout(); plt.show()

print(f"без стрижки: листьев {full.get_n_leaves():4d}, контроль {full.score(Xv, yv):.4f}")
print(f"со стрижкой: листьев {pruned.get_n_leaves():4d}, контроль {pruned.score(Xv, yv):.4f}")

> **Вывод.** Как выглядит граница решения и почему именно так? Что даёт стрижка?
>
> *(ваш ответ здесь)*

---
# Часть 3. Бэггинг: проверяем утверждение 5.12

Утверждение 5.12: если $a_1,\dots,a_B$ одинаково распределены, имеют дисперсию
$\sigma^2$ и попарную корреляцию $\rho$, то

$$
\mathrm{Var}\Bigl(\tfrac1B\sum_b a_b(x)\Bigr) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2 .
$$

Отсюда сразу всё главное про ансамбли: увеличением $B$ убирается только второе
слагаемое, а первое, $\rho\sigma^2$, — **предел**, ниже которого усреднение не
опускается. Случайный лес атакует именно $\rho$.

> **Напоминание — бэггинг.** Bagging = **b**ootstrap **agg**regat**ing**. Из обучающей выборки $B$ раз
> берут бутстреп-подвыборку (случайный выбор $\ell$ объектов **с возвращением**,
> занятие 3), на каждой обучают свой алгоритм и усредняют ответы — для регрессии
> среднее, для классификации голосование.
>
> Зачем это работает, видно из формулы выше: усреднение $B$ величин с дисперсией
> $\sigma^2$ уменьшает дисперсию, а смещение оставляет прежним. Значит, бэггинг
> имеет смысл ровно для алгоритмов с малым смещением и большим разбросом —
> то есть для **глубоких** деревьев. Отсюда и практическое правило: в бэггинге
> деревья не стригут.
>
> Случайный лес — это бэггинг плюс ещё один источник различий: в каждой вершине
> перебираются не все признаки, а случайное подмножество. Это уменьшает $\rho$ —
> корреляцию между деревьями — и тем самым снижает предел $\rho\sigma^2$.

In [ ]:
D = 5
F5 = lambda X: np.sin(2 * X[:, 0]) + 0.7 * X[:, 1] ** 2 - 0.5 * X[:, 2] * X[:, 3]


def sample5(n, g):
    X = g.uniform(-2, 2, size=(n, D))
    return X, F5(X) + g.normal(0, 0.35, n)


# Случайность ОБУЧАЮЩЕЙ ВЫБОРКИ -- она и создаёт корреляцию между деревьями
x0 = np.array([[0.7, -0.4, 1.1, 0.2, -1.3]])
B_MAX, N_REP = 30, 200
per_tree = np.empty((N_REP, B_MAX))
for r in range(N_REP):
    gb = np.random.default_rng(30_000 + r)
    Xs, ys = sample5(150, gb)                       # своя выборка на повторение
    for b in range(B_MAX):
        idx = gb.integers(0, len(ys), len(ys))      # бутстреп внутри выборки
        per_tree[r, b] = DecisionTreeRegressor(random_state=b).fit(Xs[idx], ys[idx]).predict(x0)[0]

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# per_tree -- матрица (N_REP, B_MAX): предсказание дерева b в повторении r
# TODO (3 строки):
#   sigma^2 -- средняя по деревьям дисперсия предсказания (по оси повторений);
#   rho     -- средняя попарная корреляция столбцов (np.corrcoef, без диагонали);
#   emp     -- измеренная дисперсия среднего по первым B деревьям, B = 1..B_MAX.
sigma2, rho = ..., ...
Bs = np.arange(1, B_MAX + 1)
emp = ...
theory = rho * sigma2 + (1 - rho) / Bs * sigma2

In [ ]:
fig, ax = plt.subplots()
ax.plot(Bs, emp, "o", ms=5, label="измеренный разброс среднего")
ax.plot(Bs, theory, lw=2, label=r"теория: $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$")
ax.axhline(rho * sigma2, ls="--", color="black", label=r"предел $\rho\sigma^2$")
ax.set_xlabel("число деревьев $B$"); ax.set_ylabel(r"$\mathrm{Var}(\bar a(x_0))$")
ax.set_title("Утверждение 5.12 на эксперименте"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Смещение и разброс: дерево / бэггинг / случайный лес
g_eval = np.random.default_rng(123)
X_eval, _ = sample5(200, g_eval)
f_eval = F5(X_eval)

rows = []
for lbl, mf in [("одиночное дерево", None), ("бэггинг (все признаки)", 1.0),
                ("случайный лес (40% признаков)", 0.4)]:
    preds = np.empty((150, len(X_eval)))
    for r in range(150):
        g = np.random.default_rng(20_000 + r)
        Xs, ys = sample5(150, g)
        est = (DecisionTreeRegressor(random_state=r) if mf is None
               else RandomForestRegressor(n_estimators=25, max_features=mf, random_state=r))
        preds[r] = est.fit(Xs, ys).predict(X_eval)
    b2, v = np.mean((f_eval - preds.mean(0)) ** 2), np.mean(preds.var(0))
    rows.append({"модель": lbl, "смещение^2": b2, "разброс": v, "сумма": b2 + v})
display(pd.DataFrame(rows).set_index("модель").round(4))

> **Вывод.** Согласуется ли формула с экспериментом? Зачем случайный лес ограничивает число признаков при расщеплении, если это ухудшает каждое отдельное дерево?
>
> *(ваш ответ здесь)*

---
# Часть 4. Лес против бустинга

Бэггинг усредняет **независимо обученные** алгоритмы, поэтому от числа деревьев
не переобучается. Бустинг строит **зависимую** последовательность: каждый
следующий алгоритм учится на ошибках предыдущих, и после некоторого $T$
начинается переобучение. Разница видна на одном графике.

Отсюда и противоположные требования к базовому алгоритму: бэггингу нужен
алгоритм с малым смещением и большим разбросом (глубокое дерево), бустингу —
наоборот, слабый: обычно **пень** (дерево глубины 1, одно расщепление) или
дерево глубины 2–3. Бустинг снижает смещение, накапливая его по шагам, а
разброс у него и так мал — усиливать его глубокими деревьями нельзя.

In [ ]:
n_grid = [1, 2, 5, 10, 25, 50, 100, 200, 400]
fig, ax = plt.subplots()
for name, factory in [
        ("RandomForest", lambda n: RandomForestClassifier(n_estimators=n, random_state=0)),
        ("AdaBoost", lambda n: AdaBoostClassifier(n_estimators=n, random_state=0)),
        ("GradientBoosting", lambda n: GradientBoostingClassifier(n_estimators=n,
                                                                  random_state=0))]:
    tr, te = [], []
    for n in n_grid:
        m = factory(n).fit(Xa, ya)
        tr.append(m.score(Xa, ya)); te.append(m.score(Xv, yv))
    line, = ax.plot(n_grid, te, "o-", lw=2, label=f"{name} (контроль)")
    ax.plot(n_grid, tr, ":", lw=1.3, color=line.get_color(), label=f"{name} (обучение)")
ax.set_xscale("log"); ax.set_xlabel("число базовых алгоритмов"); ax.set_ylabel("точность")
ax.set_title("Качество против числа деревьев"); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

> **Вывод.** Какие ансамбли выходят на плато, а какие переобучаются с ростом числа деревьев? Как это следует из устройства методов?
>
> *(ваш ответ здесь)*

---
# Часть 5. Важность признаков: встроенная врёт

Два способа измерить важность:

* **по приросту** (`feature_importances_`) — суммарный взвешенный прирост
  информации по всем расщеплениям, использующим признак; вычисляется бесплатно;
* **перестановочная** — падение качества **на не участвовавших в обучении**
  объектах после случайной перестановки значений признака.

Проверим их на данных, где правильный ответ известен заранее.

In [ ]:
from sklearn.inspection import permutation_importance

gz = np.random.default_rng(RANDOM_STATE)
n_obj = 800
X_imp = np.column_stack([
    gz.normal(size=n_obj),        # полезный вещественный
    gz.integers(0, 2, n_obj),     # полезный бинарный
    gz.normal(size=n_obj),        # ШУМ, много уникальных значений
    gz.integers(0, 2, n_obj),     # ШУМ, бинарный
    gz.integers(0, 5, n_obj)])    # ШУМ, 5 категорий
names = ["полезный веществ.", "полезный бинарн.", "ШУМ веществ.",
         "ШУМ бинарн.", "ШУМ 5 катег."]
y_imp = (2.0 * X_imp[:, 0] + 1.5 * X_imp[:, 1] + gz.normal(0, 0.5, n_obj) > 1.0).astype(int)

Xia, Xiv, yia, yiv = train_test_split(X_imp, y_imp, test_size=0.35, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xia, yia)

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# TODO (2-3 строки): посчитайте важности двумя способами --
#   rf.feature_importances_ и permutation_importance(rf, Xiv, yiv, n_repeats=30),
#   сведите в таблицу с индексом names и выведите суммарную важность
#   трёх ШУМОВЫХ признаков по каждому способу.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
tab.plot.barh(ax=ax)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("важность"); ax.set_title("Два способа измерить важность признака")
plt.tight_layout(); plt.show()

> **Вывод.** Какую важность получили заведомо бесполезные признаки? Почему шумовой вещественный «важнее» шумового бинарного?
>
> *(ваш ответ здесь)*

---
# Часть 6. Своя выборка

In [ ]:
from sklearn.metrics import roc_auc_score

data = load_personal(variant)
Xtr, Xte, ytr, yte = data["X_train"], data["X_test"], data["y_train"], data["y_test"]
if data["task"] == "regression":
    thr = np.median(ytr)
    ytr, yte = (ytr > thr).astype(int), (yte > thr).astype(int)
ytr, yte = ytr.astype(int), yte.astype(int)

rows = []
for name, m in [("дерево (глубина 3)", DecisionTreeClassifier(max_depth=3, random_state=0)),
                ("дерево (без ограничений)", DecisionTreeClassifier(random_state=0)),
                ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=0)),
                ("GradientBoosting", GradientBoostingClassifier(random_state=0))]:
    m.fit(Xtr, ytr)
    rows.append({"модель": name, "AUC на контроле": roc_auc_score(yte, m.predict_proba(Xte)[:, 1])})
display(pd.DataFrame(rows).set_index("модель").round(4))

In [ ]:
best = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
perm_own = permutation_importance(best, Xte, yte, n_repeats=15, random_state=0,
                                  scoring="roc_auc")
imp = pd.Series(perm_own.importances_mean, index=data["feature_names"]).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 4.5))
imp[::-1].plot.barh(ax=ax, color="#128C7E")
ax.set_xlabel("падение ROC-AUC при перестановке признака")
ax.set_title("Что определяет предсказание")
plt.tight_layout(); plt.show()

> **Вывод.** Согласуется ли важность признаков с вашими ожиданиями от предметной области? Что делать, если в топ попал технический признак?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Почему дерево не может провести диагональную границу и что с этим делают на практике? Назовите два способа.
2. В бэггинге деревья намеренно не стригут, а в бустинге берут пни глубины 1–3. Объясните оба решения через разложение смещение–разброс.
3. Из утверждения 5.12 следует, что $\mathrm{Var}(\bar a)\to\rho\sigma^2$. Какими способами случайный лес уменьшает $\rho$ и чем за это платит?
4. Ваш лес показал важность 0.22 у признака «номер договора». Что проверите первым делом?

---

**Дома:** откройте `lab06_homework.ipynb` — там две задачи: своё решающее дерево и свой AdaBoost по теореме 5.15.